# 🏠 German Rental Price Classification: Modeling
**Input:** `data/processed/immo_clean.csv` (from `01_data_preparation.ipynb`)
**Goal:** predict `rentClass` (Low / Medium / High per m²) from a flat's features. Target: accuracy ≥ 70%, few Low ↔ High mistakes.

## Setup

In [23]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df_clean = pd.read_csv("../data/processed/immo_clean.csv")
df_clean["rentClass"] = pd.Categorical(df_clean["rentClass"],
                                       categories=["Low", "Medium", "High"], ordered=True)
print("Shape:", df_clean.shape)

Shape: (267855, 26)


## 7. Data Preparation for Modeling
### 7.1 Choose features (X) and target (y)
X = all features except `rentClass` (target), `rentPerSqm` and `baseRent` (leakage).
Weak features (garden, cellar) are kept; feature importance in Phase 9 will judge them.


In [24]:
not_features=["rentClass", "rentPerSqm", "baseRent"]
X=df_clean.drop(columns=not_features)
y=df_clean["rentClass"]   

print("X shape:", X.shape)
print("y shape:", y.shape)
print("\nFeatures:", list(X.columns))

X shape: (267855, 23)
y shape: (267855,)

Features: ['regio1', 'heatingType', 'newlyConst', 'balcony', 'picturecount', 'pricetrend', 'yearConstructed', 'hasKitchen', 'cellar', 'livingSpace', 'condition', 'interiorQual', 'petsAllowed', 'lift', 'typeOfFlat', 'noRooms', 'thermalChar', 'floor', 'numberOfFloors', 'garden', 'regio2', 'hasParking', 'wasRefurbished']


### 7.2 Train / test split
80% train (the model learns from it), 20% test (hidden until the final evaluation).
`stratify=y` keeps the 33/33/33 class mix in both sets; `random_state=42` makes the split reproducible.


In [25]:
from sklearn.model_selection import train_test_split

# for random_state, you can use any number; the important thing is to use the same number every time you run the code, so you get the same split of data
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

print("Train:", X_train.shape, "  Test:", X_test.shape)
# Check: is the class mix the same in both sets?
check = pd.DataFrame({
    "train_%": (y_train.value_counts(normalize=True) * 100).round(1),
    "test_%":  (y_test.value_counts(normalize=True) * 100).round(1),
})
check


Train: (214284, 23)   Test: (53571, 23)


,train_%,test_%
rentClass,,
Low,33.4,33.4
High,33.3,33.3
Medium,33.3,33.3


### 7.3 Preprocessing pipeline
- **Numbers:** fill gaps with the median (learned from train) → scale (mean 0, spread 1).
- **Categories:** one-hot encoding (one yes/no column per category; unknown categories → all zeros).
- **Yes/no:** kept as they are (already 0/1).
- **Rule:** `fit` learns from TRAIN only; `transform` applies it to train and test.


In [26]:
num_cols  = ["yearConstructed", "livingSpace", "noRooms", "floor",
             "numberOfFloors", "thermalChar", "picturecount", "pricetrend"]
cat_cols  = ["regio1", "regio2", "heatingType", "condition",
             "interiorQual", "petsAllowed", "typeOfFlat"]
bool_cols = ["newlyConst", "balcony", "hasKitchen", "cellar",
             "lift", "garden", "hasParking", "wasRefurbished"]

# Safety check: is every feature in exactly one group?
assigned = num_cols + cat_cols + bool_cols
print("Features in X:      ", X.shape[1])
print("Features assigned:  ", len(assigned))
print("Forgotten features: ", [c for c in X.columns if c not in assigned])


Features in X:       23
Features assigned:   23
Forgotten features:  []


In [27]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Station 1: number columns → fill gaps, then scale
num_pipe = Pipeline([
    ("fill",  SimpleImputer(strategy="median")),   # empty → median (learned from train)
    ("scale", StandardScaler()),                   # mean 0, spread 1

    # shifts and stretches each column so that its average becomes 0 and its typical spread becomes 1.
])

# Station 2: category columns → one yes/no column per category
cat_pipe = OneHotEncoder(handle_unknown="ignore")
# handle_unknown="ignore" means: if a test flat has a category never seen in training, give it all zeros instead of crashing.
# Why don't categories need gap-filling? We already replaced their empty cells with "unknown" in step 4.3, and that became a category like any other.
# Join the stations: each group of columns goes to its own station

# Build the splitter called preprocessor. Inside is a list of stations, each written as ("name", tool, which_columns).
preprocessor = ColumnTransformer([
    ("num",  num_pipe,      num_cols),
    ("cat",  cat_pipe,      cat_cols),
    ("bool", "passthrough", bool_cols),            # "passthrough" = keep unchanged because T/F they are already 0/1.
])


fit = LEARN something from the data and remember it.
transform = USE what was learned to change data. happen on train and test 
transform applies what was learned to both the training and the test data, not only the test data


What does "apply" (transform) mean?
Apply = actually CHANGE the values in the table, using what fit measured.

fit only measures and changes nothing. transform does the actual work on the table.
for test data we use transform to apply what learnt form train into the testing

In [28]:
X_train_ready = preprocessor.fit_transform(X_train)   # LEARN from train, then apply
X_test_ready  = preprocessor.transform(X_test)        # only APPLY to test

print("Train before:", X_train.shape, "→ after:", X_train_ready.shape)
print("Test  before:", X_test.shape,  "→ after:", X_test_ready.shape)


Train before: (214284, 23) → after: (214284, 220)
Test  before: (53571, 23) → after: (53571, 220)


## 8. Modeling
### 8.0 Validation set
The test set is kept for the final evaluation (Phase 9). To compare models now, 20% of the TRAIN set is used as a **validation set** (a practice exam) we choose the models based on it and keep the test for final score to give the honest score.


In [29]:
# make new split from X_train for validation
X_tr,X_val,y_tr,y_val=train_test_split(X_train,y_train,test_size=0.2,random_state=42,stratify=y_train)

print("Train part:", X_tr.shape, " Validation:", X_val.shape, " Test (untouched):", X_test.shape)


Train part: (171427, 23)  Validation: (42857, 23)  Test (untouched): (53571, 23)


In Phase 8 we'll try many models (baseline, Decision Tree, Random Forest, k-NN…). For each one we must do the same 2 jobs:

Build it: attach the model to our preparation machine.
Test it: train it, then score it on train and on validation.

In [37]:
# Upgraded helpers (replace the old versions):
# - dense=True  → the preparation machine outputs a normal table (needed by Naive Bayes it cant handle spare tables that store only the non-zero values)
# - sample=N    → the model learns from N random flats only (for slow models like k-NN very slow on 171k flats)

def make_model(model, dense=False):
    prep = clone(preprocessor)
    if dense:
        prep.set_params(sparse_threshold=0)      # output a normal (dense) table
    return Pipeline([("prep", prep), ("model", model)])

def evaluate(pipe, name, sample=None):
    if sample:
        X_fit = X_tr.sample(sample, random_state=42)   #pick subset of random flats from the whole training set (same ones every run)
        y_fit = y_tr.loc[X_fit.index]                  #locate their answers
    else:
        X_fit, y_fit = X_tr, y_tr

    pipe.fit(X_fit, y_fit)
    train_acc = accuracy_score(y_fit, pipe.predict(X_fit)) * 100
    val_acc   = accuracy_score(y_val, pipe.predict(X_val)) * 100
    print(f"{name}: train {train_acc:.1f}% | validation {val_acc:.1f}% | gap {train_acc - val_acc:.1f}")
    return val_acc


### 8.1 Baseline
A model that always predicts the most common class. Every real model must clearly beat it.

The baseline is the simplest possible "model": it ignores all the features and always predicts the most common class. In textbooks it's called the "Zero Rule" (ZeroR) or majority-class baseline, because it uses zero rules about the data.

With our balanced classes, it gets about 33%.

In [33]:
from sklearn.dummy import DummyClassifier

baseline = make_model(DummyClassifier(strategy="most_frequent"))
evaluate(baseline, "Baseline")


Baseline: train 33.4% | validation 33.4% | gap 0.0


33.36211120703736

100% ─────────────────────────
       ↑ our goal: ≥ 70%  (business need, set in Phase 1)
 70% ─────────────────────────
       ↑ better than nothing, but not good enough
 33% ───── BASELINE (floor) ──
       ↓ worse than learning nothing → something is broken
  0% ─────────────────────────


### 8.2 Decision Tree


In [ ]:
from sklearn.tree import DecisionTreeClassifier
# random_state is a number that makes the model's random choices repeatable. You can use any number, but always use the same one so you get the same results every time you run the code.
tree = make_model(DecisionTreeClassifier(random_state=42)) 
evaluate(tree, "Decision Tree")

Decision Tree: train 100.0% | validation 71.1% | gap 28.9


71.09923699745666

**Finding:** the unlimited tree scores 100% on train but 71.1% on validation (gap 28.9) → strong **overfitting** (it memorizes the training flats). It still beats the baseline and reaches the 70% goal.
Next: limit the depth so the tree must learn general patterns.


In [35]:
for depth in [5, 10, 15, 20, 25]:
    tree_d = make_model(DecisionTreeClassifier(max_depth=depth, random_state=42))
    evaluate(tree_d, f"Tree depth {depth}");


Tree depth 5: train 64.8% | validation 64.3% | gap 0.6
Tree depth 10: train 70.4% | validation 69.2% | gap 1.1
Tree depth 15: train 76.6% | validation 71.8% | gap 4.8
Tree depth 20: train 83.3% | validation 72.4% | gap 10.9
Tree depth 25: train 89.3% | validation 72.9% | gap 16.3


### 8.3 Random Forest
Many different trees (random samples of flats and features) vote on each flat. Individual trees' mistakes cancel out, so the forest usually generalizes better than one tree.


In [36]:
from sklearn.ensemble import RandomForestClassifier

forest = make_model(RandomForestClassifier(
    n_estimators=100,     # number of trees in the forest
    n_jobs=-1,            # use all CPU cores (faster)
    random_state=42,      # reproducible
))
evaluate(forest, "Random Forest");


Random Forest: train 100.0% | validation 79.4% | gap 20.6


**Random Forest:** train 100.0% | validation **79.4%** | gap 20.6
- +8.3 points over the single tree (71.1%), and well above the 70% goal.
- The 100% train score is normal for forests: each deep tree memorizes its own sample, so training flats are "remembered". The gap is therefore less meaningful than for a single tree.
- Overfitting only matters if it hurts unseen data. Here validation improved strongly, because the trees memorize different accidents and the vote cancels them out (wisdom of the crowd).
- **Rule:** judge models by validation score; limiting the trees will be tested in tuning (8.6).


### 8.4 More models (one from each family, from the course material)
| Model | Family | Idea |
|---|---|---|
| Logistic Regression | Linear (formula) | Adds up weighted clues into a score; assumes straight-line effects |
| k-NN | Distance | Copies the class of the 15 most similar flats (trained on a 30k sample because it is slow) |
| Naive Bayes | Probability | Combines the probability of each clue, assuming clues are independent |
| Bagging | Ensemble | Many trees on random samples of flats vote; like Random Forest but without random feature subsets |


In [38]:
# Validation scores of the models trained so far (8.1–8.3)
results = {
    "Baseline":      33.4,
    "Decision Tree": 71.1,   # ← replace with your BEST depth's validation score from 8.2
    "Random Forest": 79.4,
}

In [39]:
from sklearn.linear_model import LogisticRegression

results["Logistic Regression"] = evaluate(
    make_model(LogisticRegression(max_iter=1000)),   # max_iter: enough rounds to finish learning
    "Logistic Regression")


Logistic Regression: train 74.7% | validation 74.1% | gap 0.6


In [ ]:
from sklearn.neighbors import KNeighborsClassifier

results["k-NN (30k sample)"] = evaluate(
    make_model(KNeighborsClassifier(n_neighbors=15, n_jobs=-1)),   # 15 most similar flats vote small k overfittinf and big cause underfitting ,n_jobs=-1: use all processor cores
    "k-NN", sample=30000)

k-NN: train 74.6% | validation 70.2% | gap 4.4


In [41]:
from sklearn.naive_bayes import GaussianNB

# GaussianNB (for number features); the course used CategoricalNB, which only handles categories
results["Naive Bayes"] = evaluate(make_model(GaussianNB(), dense=True), "Naive Bayes")


Naive Bayes: train 62.4% | validation 62.3% | gap 0.1


In [43]:
from sklearn.ensemble import BaggingClassifier
# bagging: make many random training samples,train 1 tree on each,repeat with other 50 trees, vote and take the majority vote
results["Bagging"] = evaluate(
    make_model(BaggingClassifier(
        estimator=DecisionTreeClassifier(random_state=42),   # the model copied many times
        n_estimators=50,                                     # 50 trees
        n_jobs=-1,
        random_state=42,
    )),
    "Bagging")


Bagging: train 100.0% | validation 78.3% | gap 21.7


Random Forest = Bagging with trees + random feature subsets at each split.
Remove the feature randomness, and you get Bagging.


Which models does bagging help?
Bagging works by averaging out mistakes that come from the training data. It only helps if the model changes a lot when the training data changes slightly. Such a model is called unstable (or high-variance).

Model	Stable or unstable?	Does bagging help?
Decision Tree (deep)	Unstable: a few different flats can produce a very different tree	✅ A lot. That's why bagging is almost always used with trees.
Logistic Regression	Stable: a few different flats barely change the weights	❌ Hardly. 50 copies would be almost identical, and so would their votes.
k-NN (k = 15)	Fairly stable: 15 neighbours already average things out	❌ Little
Naive Bayes	Very stable: it just counts probabilities	❌ Almost none

#### Voting 
Bagging only helps unstable models (trees), and Random Forest already covers bagged trees.
Voting combines **different model families** (formula, questions, probabilities), so their mistakes differ and can cancel out.


In [44]:
from sklearn.ensemble import VotingClassifier

voting = VotingClassifier(
    estimators=[
        ("logreg", LogisticRegression(max_iter=1000)),                      # formula
        ("tree",   DecisionTreeClassifier(max_depth=20, random_state=42)),  # questions
        ("nb",     GaussianNB()),                                           # probabilities
    ],
    voting="hard",     # each model gets one vote; the majority class wins
    n_jobs=-1,         # train the 3 models at the same time
)

results["Voting"] = evaluate(make_model(voting, dense=True), "Voting")


Voting: train 78.6% | validation 74.0% | gap 4.7


**Findings (model comparison on validation):**
- **Best model: Random Forest (79.4%)**, well above the 70% goal.
- **Random Forest vs Bagging: 79.4% vs 78.3%** → the random feature subsets help (+1.1): trees become more different, so their mistakes cancel better.
- **Logistic Regression (74.1%) beats the single tree (72.4%)** with almost no gap (0.6): a stable model that captures the strong location effects well, but its straight-line assumption limits it (it cannot capture the U-shape of `livingSpace` or the location-dependent garden effect), so it stays below the tree ensembles.
- **Voting (74.0%) did not beat its best member** (Logistic Regression 74.1%): the weak Naive Bayes voter pulled it down. Combining models only helps when the voters are individually good **and** make different mistakes.
- **k-NN (70.2%)**: just reaches the goal; trained on a 30k sample for speed, so its score may be slightly underestimated.
- **Naive Bayes (62.3%)**: the weakest and **below the 70% goal**. It assumes features are independent, which is false here (e.g. `noRooms` & `livingSpace`, `regio1` & `regio2`).
- **All models beat the baseline (33.4%).** Six of seven reach 70% (all except Naive Bayes).
- **Lessons:** ensembles of trees win on this data; bagging helps unstable models; a vote is only as good as its voters.
